# Contract

A contract is the base concept from which you can calculate energy costs.
It consists of 4 components:
- a `provider` tariff
- a `distributor` tariff
- `fees` — which are essentially government tariffs — multiple fee tariffs can be provided as a list and will be merged automatically
- a `tax_rate` (a percentage applied to all tariff costs)

Given a contract and consumption data, you can calculate the cost of that consumption under the terms of the contract.

The output DataFrame uses a structured MultiIndex with fixed, translatable labels based on `TariffCategory` and `CostGroup` enums, making it predictable and easy to use for i18n.

In [ ]:
from zoneinfo import ZoneInfo

import pandas as pd

from energy_cost import Contract, Meter, Tariff
from energy_cost.data import CustomerType
from energy_cost.data.be.flanders.electricity import data

contract = Contract(
    provider=Tariff.from_yaml("../examples/tariffs/fixed.yml"),
    distributor=data.distributors["fluvius_imewo"],
    fees=data.fees[CustomerType.RESIDENTIAL],
    taxes=data.taxes,
    timezone=ZoneInfo("Europe/Brussels"),
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption])

timestamp    provider        distributor              \
                            consumption  total    capacity consumption   
                                  total  total       total       total   
0 2024-01-01 00:00:00+01:00       59.52  59.52    8.209764   23.676520   
1 2024-02-01 00:00:00+01:00       55.68  55.68    8.209764   22.149003   
2 2024-03-01 00:00:00+01:00        0.02   0.02    8.209764    0.007956   

                                                       fees             \
             fixed                total         consumption              
  data_manangement     total      total energy_contribution     excise   
0         1.209508  1.209508  33.095793            1.146415  28.260096   
1         1.131475  1.131475  31.490243            1.072452  26.436864   
2         1.207883  1.207883   9.425603            0.000385   0.009496   

                              total     taxes  
                  total       total     total  
       total      total       total     total  
0  29.406511  29.406511  129.343642  7.321338  
1  27.509316  27.509316  121.560333  6.880774  
2   0.009881   0.009881   10.022813  0.567329

> **Note**: most tariffs are based on indexes, make sure to define all required indexes before calculating costs.
>
> See the [index documentation](./index.ipynb) for more details.
>
> If you are defining your own tariffs, also have a look at the [tariff documentation](./tariff.ipynb) for details on how to implement tariffs.
>
> The [tax documentation](./tax.ipynb) can also be usefull if you want to define your own taxes.

### Resolution

By default the cost is calculated for each month, but you can specify a different resolution if you want to calculate costs for different time periods eg yearly.

In [2]:
import isodate

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption], resolution=isodate.Duration(years=1))

timestamp    provider                          distributor  \
                            consumption     fixed          total    capacity   
                                  total fixed_fee  total   total       total   
0 2024-01-01 00:00:00+01:00      702.72       0.0    0.0  702.72   98.517173   
1 2025-01-01 00:00:00+01:00      630.72     120.0  120.0  750.72  133.105596   
2 2026-01-01 00:00:00+01:00      135.96     180.0  180.0  315.96   33.875614   

                                                                 fees  \
  consumption            fixed              total         consumption   
        total data_manangement  total       total energy_contribution   
0  279.535692            14.28  14.28  392.332864           13.535090   
1  412.792925            17.51  17.51  563.408521           13.498109   
2   59.240491            17.85  17.85  110.966105            2.182271   

                                             total      taxes  
                                total        total      total  
       excise       total       total        total      total  
0  333.651456  347.186546  347.186546  1528.773775  86.534365  
1  332.739840  346.237949  346.237949  1759.988458  99.621988  
2   53.794840   55.977111   55.977111   511.877409  28.974193

### Time range

By default start and end are set to the first and last timestamp in the data, but you can specify a different start and end if you want to calculate costs for a different time period than the one covered by the data.

This is useful for the Flemish capacity tariff that uses a rolling average of the consumption of the last 12 months to determine the cost for the next month. In this case you would need to provide data for at least 12 months before the start of the period you want to calculate costs for.

In [3]:
import datetime as dt
from zoneinfo import ZoneInfo

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": [0.002] * 4 * 24 * 365 + [0.003] * 4 * 24 * 365 + [0.003] * 4 * 24 * 60 + [0.003],
        }
    )
)

contract.calculate_cost(
    [consumption],
    start=dt.datetime(2025, 1, 1, tzinfo=ZoneInfo("Europe/Brussels")),
    end=dt.datetime(2025, 12, 31, tzinfo=ZoneInfo("Europe/Brussels")),
)

timestamp    provider                         distributor  \
                             consumption     fixed         total    capacity   
                                   total fixed_fee total   total       total   
0  2025-01-01 00:00:00+01:00      803.52      10.0  10.0  813.52   11.092133   
1  2025-02-01 00:00:00+01:00      725.76      10.0  10.0  735.76   11.092133   
2  2025-03-01 00:00:00+01:00      802.44      10.0  10.0  812.44   11.092133   
3  2025-04-01 00:00:00+02:00      777.60      10.0  10.0  787.60   11.092133   
4  2025-05-01 00:00:00+02:00      803.52      10.0  10.0  813.52   11.092133   
5  2025-06-01 00:00:00+02:00      777.60      10.0  10.0  787.60   11.104060   
6  2025-07-01 00:00:00+02:00      803.52      10.0  10.0  813.52   11.138350   
7  2025-08-01 00:00:00+02:00      803.52      10.0  10.0  813.52   11.192971   
8  2025-09-01 00:00:00+02:00      777.60      10.0  10.0  787.60   11.266127   
9  2025-10-01 00:00:00+02:00      804.60      10.0  10.0  814.60   11.356231   
10 2025-11-01 00:00:00+01:00      777.60      10.0  10.0  787.60   11.461871   
11 2025-12-01 00:00:00+01:00      803.52      10.0  10.0  813.52   11.461871   

                                                                     fees  \
   consumption            fixed                 total         consumption   
         total data_manangement     total       total energy_contribution   
0   525.886877         1.487151  1.487151  538.466160           17.196221   
1   474.994598         1.343233  1.343233  487.429964           15.532070   
2   525.180040         1.485152  1.485152  537.757324           17.173108   
3   508.922784         1.439178  1.439178  521.454095           16.641504   
4   525.886877         1.487151  1.487151  538.466160           17.196221   
5   508.922784         1.439178  1.439178  521.466022           16.641504   
6   525.886877         1.487151  1.487151  538.512378           17.196221   
7   525.886877         1.487151  1.487151  538.566998           17.196221   
8   508.922784         1.439178  1.439178  521.628089           16.641504   
9   526.593714         1.489150  1.489150  539.439095           17.219334   
10  508.922784         1.439178  1.439178  521.823833           16.641504   
11  525.886877         1.487151  1.487151  538.835898           17.196221   

                                              total       taxes  
                                 total        total       total  
        excise       total       total        total       total  
0   406.114744  423.310965  423.310965  1881.814953  106.517828  
1   366.813317  382.345388  382.345388  1701.867473   96.332121  
2   405.568891  422.741999  422.741999  1879.315682  106.376359  
3   393.014268  409.655772  409.655772  1821.832460  103.122592  
4   406.114744  423.310965  423.310965  1881.814953  106.517828  
5   393.014268  409.655772  409.655772  1821.845102  103.123308  
6   406.114744  423.310965  423.310965  1881.863943  106.520601  
7   406.114744  423.310965  423.310965  1881.921841  106.523878  
8   393.014268  409.655772  409.655772  1822.016894  103.133032  
9   406.660597  423.879931  423.879931  1884.594168  106.675142  
10  393.014268  409.655772  409.655772  1822.224382  103.144776  
11  406.114744  423.310965  423.310965  1882.206875  106.540012

As you can see in the above example, while the capacity was constant in 2025, the first months are still cheaper then the last, as the cost of the capacity tariff is based on the consumption of the previous 12 months, which includes the cheaper months of 2024.

### Injection

Aside from a consumption timeseries, you can also provide injection as a separate timeseries, as these are often measured independently and have different tariffs. If you provide injection data, the cost of the injection will be calculated separately and subtracted from the cost of the consumption, as injection is essentially negative consumption.

In [4]:
from energy_cost import PowerDirection

injection = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    ),
    direction=PowerDirection.INJECTION,
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption, injection], resolution=isodate.Duration(years=1))

timestamp    provider                                      \
                            consumption     fixed        injection    total   
                                  total fixed_fee  total     total    total   
0 2024-01-01 00:00:00+01:00      702.72       0.0    0.0  -140.544  562.176   
1 2025-01-01 00:00:00+01:00      630.72     120.0  120.0  -105.120  645.600   
2 2026-01-01 00:00:00+01:00      135.96     180.0  180.0   -28.325  287.635   

  distributor                                                  \
     capacity consumption            fixed              total   
        total       total data_manangement  total       total   
0   98.517173  279.535692            14.28  14.28  392.332864   
1  133.105596  412.792925            17.51  17.51  563.408521   
2   33.875614   59.240491            17.85  17.85  110.966105   

                 fees                                            total  \
          consumption                               total        total   
  energy_contribution      excise       total       total        total   
0           13.535090  333.651456  347.186546  347.186546  1388.229775   
1           13.498109  332.739840  346.237949  346.237949  1654.868458   
2            2.182271   53.794840   55.977111   55.977111   483.552409   

       taxes  
       total  
       total  
0  86.534365  
1  99.621988  
2  28.974193